# Time-Frequency Analysis

### Import all the necessary functions

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
#-----imports-----#
import numpy as np
from matplotlib import pyplot as plt
import mne

from mne import Epochs, create_info
from mne.io import RawArray
from mne.time_frequency import AverageTFRArray, EpochsTFRArray, tfr_array_morlet

import os
import matplotlib.pyplot as plt

In [ ]:
import sys 
from pathlib import Path

# get project root 
project_root = Path("/Users/Tysia/Desktop/movidoc/tictrack_eeg_analysis")
sys.path.append(str(project_root))



In [ ]:
from _02_Movidoc_tictrack_prepro_TTL_extraction_patients import (
    load_data,
    extract_stimuli,
    preprocess_data, 
    apply_ICA, 
    apply_rest_reference,
    recalibrate_from_first_event,
    collect_ttl_with_phases
)

In [ ]:
from _03_Movidoc_tictrack_tic_extraction_patients import extract_tics_from_excel

In [ ]:
from _04_Movidoc_full_pipeline_analysis_patients import (
    assign_phase_to_tics,
    extract_eeg_phase_times_from_ttl,
    realign_excel_to_eeg,
    build_merged_ttl_tics,
    run_full_pipeline_for_patient,
    shift_urges_times,
    extract_random_epochs_in_phase,
    analyse_merged_ttl_tics_spontaneous,
    extract_pre_tic_epochs,
)

In [ ]:
from _05_Movidoc_tictrack_urge_extraction_function import (
    deduplicate_imitated_tics,
    analyse_merged_ttl_tics_imitated,
    deduplicate_suppressed_tics,
    analyse_merged_ttl_tics_suppressed,
)

In [ ]:
from _06_Movidoc_tictrack_TFR_functions import (
    trf_analysis_normalized,
    tfr_per_ROI_normalized,
    plot_trf_roi
)



### Load data

In [ ]:
# Base directory for this project
base_dir = "/Users/Tysia/Desktop/movidoc/tictrack_eeg_analysis"
patient_files_dir = os.path.join(base_dir, "PATIENT FILES")
eeg_dir = os.path.join(patient_files_dir, "EEG PATIENT FILES")
excel_dir = os.path.join(patient_files_dir, "EXCEL PATIENT FILES")



In [ ]:
# Define patient configurations
patients = [
    {
        "montage": "standard_1020", # montage with 32 electrodes (from DS26 to BC29 : always 32 electrodes)
        "vhdr": os.path.join(eeg_dir, "MOVIDOCTicTrack000010.vhdr"),
        "excel": os.path.join(excel_dir, "DS26_annotations_binary-table_cutted.xlsx"), # DS26
        "fps": 30,
        "min_absence_frames": 30,
        "excel_phase_times": [13.728, 50.028, 190.080, 349.272, 980.727, 1277.991]
    },
    {
        "montage": "standard_1020", # montage with 32 electrodes (from DS26 to BC29 : always 32 electrodes)
        "vhdr": os.path.join(eeg_dir, "MOVIDOCTicTrack_BB28-bis.vhdr"), #BB28
        "excel": os.path.join(excel_dir, "BB28_annotations_binary-table_cutted.xlsx"),
        "fps": 25,
        "min_absence_frames": 25,
        "excel_phase_times": [9.840, 27.320, 156.880, 302.600, 929.960, 991.760]
    },
    {
        "montage": "standard_1020", # montage with 32 electrodes (from DS26 to BC29 : always 32 electrodes)
        "vhdr": os.path.join(eeg_dir, "MOVIDOCTicTrack000013.vhdr"), # BC29
        "excel": os.path.join(excel_dir, "BC29_annotations_binary-table_cutted_Lizbeth.xlsx"),
        "fps": 30,
        "min_absence_frames": 30,
        "excel_phase_times": [17.391, 65.670, 201.597, 358.215, 1088.670, 1204.038]
    },
    {
        "montage": "standard_1005", # montage with 64 electrodes (from MM30 : always 64 electrodes)
        "vhdr": os.path.join(eeg_dir, "MOVIDOCTicTrack000030.vhdr"), # MM30
        "excel": os.path.join(excel_dir, "MM30_annotations_binary-table_cutted.xlsx"),
        "fps": 30,
        "min_absence_frames": 30,
        "excel_phase_times": [6.633, 68.277, 196.581, 323.994, 931.194, 995.049]
    },
    {
        "montage": "standard_1005", # montage with 64 electrodes (from MM30 : always 64 electrodes)
        "vhdr": os.path.join(eeg_dir, "MOVIDOCTicTrack000031.vhdr"), # SC31
        "excel": os.path.join(excel_dir, "SC31_annotations_binary-table_cutted.xlsx"),
        "fps": 30,
        "min_absence_frames": 30,
        "excel_phase_times": [12.243, 62.040, 191.664, 345.972, 970.200, 1074.546]
    }
]

## Run full pipeline - Epoch extraction

In [ ]:
def full_pipeline_extract_pre_tic_epochs(patient, phase, nb_random_epochs):
    """Extract pre-tic epochs. Returns differ by phase:
    - spontaneous: (pre_tic_epochs, random_epochs, bad_channels)
    - imitated/suppressed: (pre_tic_category1, pre_tic_category2, random_epochs, bad_channels)
    
    Returns None for any epoch category that is empty.
    """
    # Setup
    results = run_full_pipeline_for_patient(
        vhdr_path=patient["vhdr"], excel_path=patient["excel"],
        fps=patient["fps"], min_absence_frames=patient["min_absence_frames"],
        montage_name=patient["montage"], excel_phase_times=patient["excel_phase_times"]
    )
    
    phase_map = {
        "spontaneous_phase": ("start_spont", "end_spont", analyse_merged_ttl_tics_spontaneous),
        "imitated_phase": ("start_imit", "end_imit", analyse_merged_ttl_tics_imitated),
        "suppressed_phase": ("start_ret", "end_ret", analyse_merged_ttl_tics_suppressed),
    }
    
    filter_dict = {
        'start_spont': 9, 'end_spont': 10, 'start_imit': 11, 'end_imit': 12,
        'start_ret': 13, 'end_ret': 14, 'start_open': 7, 'end_open': 8,
    }
    
    phase_start, phase_end, analyse_func = phase_map[phase]
    
    def make_epochs(urge_list, pre_sec=3.0, post_sec=2):
            if not urge_list:
                return None
            urges_shifted = shift_urges_times(urge_list, results["stim2_time_original"])
            if not urges_shifted:
                return None
            return extract_pre_tic_epochs(results["raw_cropped"], urges_shifted, pre_sec, post_sec)

    # Analyze tics
    analysis_result = analyse_func(
        merged_ttl_tics=results["merged_ttl_tics"],
        phase_start_key=phase_start,
        phase_end_key=phase_end
    )
    
    # Extract random epochs (common for all phases)
    ttl_info = results["ttl"]
    start_open = next(t["time"] for t in ttl_info if int(t["ttl_name"].split()[-1]) == filter_dict["start_open"])
    end_open = next(t["time"] for t in ttl_info if int(t["ttl_name"].split()[-1]) == filter_dict["end_open"])
    
    random_epochs = extract_random_epochs_in_phase(
        results["raw_cropped"], start_open, end_open,
        nb_random_epochs, epoch_duration=3,
        event_id=999, seed=42
    )
    
    # Phase-specific returns
    if phase == "spontaneous_phase":
        print("Processing spontaneous phase...")
        pre_tic_epochs = make_epochs(analysis_result, 3.0, 2.0)
        if pre_tic_epochs is None:
            print("WARNING: No spontaneous epochs found")
        return pre_tic_epochs, random_epochs, results["bad_channels"]
    
    else:
        # Imitated or suppressed (both return two lists)
        list1, list2 = analysis_result
        
        print(f"Processing {phase} - category 1...")
        epochs1 = make_epochs(list1)
        epochs2 = make_epochs(list2)
        if epochs1 is None:
            print(f"WARNING: No category 1 epochs found for {phase}")
        
        print(f"Processing {phase} - category 2 (real)...")
        epochs2 = make_epochs(list2)
        if epochs2 is None:
            print(f"WARNING: No category 2 (real) epochs found for {phase}")
        
        return epochs1, epochs2, random_epochs, results["bad_channels"]

In [ ]:
# try for one patient for now 
pre_tic_epochs_p1_spont, random_epochs_p1_spont, bad_channels_p1_spont = full_pipeline_extract_pre_tic_epochs(patients[4], phase="spontaneous_phase", nb_random_epochs=50)

In [ ]:
# Extract pre-tic epochs for all patients in the spontaneous phase
#patient 1
pre_tic_epochs_p1_spont, random_epochs_p1_spont, bad_channels_p1_spont = full_pipeline_extract_pre_tic_epochs(patients[0], phase="spontaneous_phase", nb_random_epochs=50)
#patient 2
pre_tic_epochs_p2_spont, random_epochs_p2_spont, bad_channels_p2_spont = full_pipeline_extract_pre_tic_epochs(patients[1], phase="spontaneous_phase", nb_random_epochs=50)
#patient 3
pre_tic_epochs_p3_spont, random_epochs_p3_spont, bad_channels_p3_spont = full_pipeline_extract_pre_tic_epochs(patients[2], phase="spontaneous_phase", nb_random_epochs=50)
#patient 4
pre_tic_epochs_p4_spont, random_epochs_p4_spont, bad_channels_p4_spont = full_pipeline_extract_pre_tic_epochs(patients[3], phase="spontaneous_phase", nb_random_epochs=50)
#patient 5
pre_tic_epochs_p5_spont, random_epochs_p5_spont, bad_channels_p5_spont = full_pipeline_extract_pre_tic_epochs(patients[4], phase="spontaneous_phase", nb_random_epochs=50)




In [ ]:
# Extract pre-tic epochs for all patients in the suppressed phase 
#patient 1
pre_tic_epochs_p1_sup ,pre_tic_epochs_p1_real_sup, random_epochs_p1_sup, bad_channels_p1_sup = full_pipeline_extract_pre_tic_epochs(patients[0], phase="suppressed_phase", nb_random_epochs=50) # no real tics in suppressed phase for patient 1, so this will return empty arrays 
#patient 2
pre_tic_epochs_p2_sup, pre_tic_epochs_p2_real_sup, random_epochs_p2_sup, bad_channels_p2_sup = full_pipeline_extract_pre_tic_epochs(patients[1], phase="suppressed_phase", nb_random_epochs=50) # 
#patient 3
pre_tic_epochs_p3_sup, pre_tic_epochs_p3_real_sup, random_epochs_p3_sup, bad_channels_p3_sup = full_pipeline_extract_pre_tic_epochs(patients[2], phase="suppressed_phase", nb_random_epochs=50)
#patient 4
pre_tic_epochs_p4_sup, pre_tic_epochs_p4_real_sup, random_epochs_p4_sup, bad_channels_p4_sup = full_pipeline_extract_pre_tic_epochs(patients[3], phase="suppressed_phase", nb_random_epochs=50)
#patient 5
pre_tic_epochs_p5_sup, pre_tic_epochs_p5_real_sup, random_epochs_p5_sup, bad_channels_p5_sup = full_pipeline_extract_pre_tic_epochs(patients[4], phase="suppressed_phase", nb_random_epochs=50)




In [ ]:
# Extract pre-tic epochs for all patients in the imitated phase
#patient 1
pre_tic_epochs_p1_imit, pre_tic_epochs_p1_real_imit, random_epochs_p1_imit, bad_channels_p1_imit = full_pipeline_extract_pre_tic_epochs(patients[0], phase="imitated_phase", nb_random_epochs=50) 
#patient 2
pre_tic_epochs_p2_imit, pre_tic_epochs_p2_real_imit, random_epochs_p2_imit, bad_channels_p2_imit = full_pipeline_extract_pre_tic_epochs(patients[1], phase="imitated_phase", nb_random_epochs=50)
#patient 3
pre_tic_epochs_p3_imit, pre_tic_epochs_p3_real_imit, random_epochs_p3_imit, bad_channels_p3_imit = full_pipeline_extract_pre_tic_epochs(patients[2], phase="imitated_phase", nb_random_epochs=50)
#patient 4
pre_tic_epochs_p4_imit, pre_tic_epochs_p4_real_imit, random_epochs_p4_imit, bad_channels_p4_imit = full_pipeline_extract_pre_tic_epochs(patients[3], phase="imitated_phase", nb_random_epochs=50) # no real tics in imitated phase for patient 4, so this will return empty arrays 
#patient 5
pre_tic_epochs_p5_imit, pre_tic_epochs_p5_real_imit, random_epochs_p5_imit, bad_channels_p5_imit = full_pipeline_extract_pre_tic_epochs(patients[4], phase="imitated_phase", nb_random_epochs=50)


In [ ]:
# Each patient's data - spontaneous phase, imitated phase, suppressed phase
patients_data = {
    "DS26": {
        "spontaneous": (pre_tic_epochs_p1_spont, random_epochs_p1_spont, bad_channels_p1_spont),
        "suppressed": (pre_tic_epochs_p1_sup, pre_tic_epochs_p1_real_sup, random_epochs_p1_sup, bad_channels_p1_sup),
        "imitated": (pre_tic_epochs_p1_imit, pre_tic_epochs_p1_real_imit, random_epochs_p1_imit, bad_channels_p1_imit),
    },
    "BB28": {
        "spontaneous": (pre_tic_epochs_p2_spont, random_epochs_p2_spont, bad_channels_p2_spont),
        "suppressed": (pre_tic_epochs_p2_sup, pre_tic_epochs_p2_real_sup, random_epochs_p2_sup, bad_channels_p2_sup),
        "imitated": (pre_tic_epochs_p2_imit, pre_tic_epochs_p2_real_imit, random_epochs_p2_imit, bad_channels_p2_imit),
    },
    "BC29": {
        "spontaneous": (pre_tic_epochs_p3_spont, random_epochs_p3_spont, bad_channels_p3_spont),
        "suppressed": (pre_tic_epochs_p3_sup, pre_tic_epochs_p3_real_sup, random_epochs_p3_sup, bad_channels_p3_sup),
        "imitated": (pre_tic_epochs_p3_imit, pre_tic_epochs_p3_real_imit, random_epochs_p3_imit, bad_channels_p3_imit),
    },
    "MM30": {
        "spontaneous": (pre_tic_epochs_p4_spont, random_epochs_p4_spont, bad_channels_p4_spont),
        "suppressed": (pre_tic_epochs_p4_sup, pre_tic_epochs_p4_real_sup, random_epochs_p4_sup, bad_channels_p4_sup),
        "imitated": (pre_tic_epochs_p4_imit, pre_tic_epochs_p4_real_imit, random_epochs_p4_imit, bad_channels_p4_imit),
    },
    "SC31": {
        "spontaneous": (pre_tic_epochs_p5_spont, random_epochs_p5_spont, bad_channels_p5_spont),
        "suppressed": (pre_tic_epochs_p5_sup, pre_tic_epochs_p5_real_sup, random_epochs_p5_sup, bad_channels_p5_sup),
        "imitated": (pre_tic_epochs_p5_imit, pre_tic_epochs_p5_real_imit, random_epochs_p5_imit, bad_channels_p5_imit),
    },
}

## Raw Data Analysis

In [ ]:
def plot_raw_epochs_with_roi(epochs, patient, patient_id, phase_name, 
                                  save_folder='epochs_with_roi_colors', 
                                  scalings='auto'):
    import os
    import matplotlib.pyplot as plt
    import mne
    
    # Define ROIs and channels
    channels_to_use_32 = ["Cz", "C3", "C4", "Pz", "Fp1", "Fp2"]
    channels_to_use_64 = ["Cz", "FCz", "C3", "FC3", "CP3", "C4", "CP4", "FC4", "Pz", "CPz", "Fp1", "Fp2", "AF3", "AF4"]
    
    roi_lists_32 = {
        "midline_premotor": ["Cz"],
        "left_sensorimotor": ["C3"],
        "right_sensorimotor": ["C4"],
        "midline_posterior": ["Pz"],
        "midline_prefrontal": ["Fp1", "Fp2"]
    }
    roi_lists_64 = {
        "midline_premotor": ["Cz", "FCz"],
        "left_sensorimotor": ["C3", "FC3", "CP3"],
        "right_sensorimotor": ["C4", "CP4", "FC4"],
        "midline_posterior": ["Pz", "CPz"],
        "midline_prefrontal": ["Fp1", "Fp2", "AF3", "AF4"]
    }
    
    # ROI colors
    roi_colors = {
        "midline_premotor": "red",
        "left_sensorimotor": "blue",
        "right_sensorimotor": "green",
        "midline_posterior": "purple",
        "midline_prefrontal": "orange"
    }
    
    # Select channels and ROIs based on montage
    if patient["montage"] == "standard_1020":
        channels_to_use = channels_to_use_32
        roi_lists = roi_lists_32
    elif patient["montage"] == "standard_1005":
        channels_to_use = channels_to_use_64
        roi_lists = roi_lists_64
    else:
        raise ValueError(f"Unknown montage: {patient['montage']}")
    
    # Pick ROI channels
    available_channels = [ch for ch in channels_to_use if ch in epochs.ch_names]
    if not available_channels:
        print(f"No ROI channels available in epochs")
        return
    
    epochs_roi = epochs.copy().pick(available_channels)
    
    # Create channel-to-color mapping
    channel_colors = {}
    for roi_name, roi_channels in roi_lists.items():
        for ch in roi_channels:
            if ch in available_channels:
                channel_colors[ch] = roi_colors[roi_name]
    
    # Create save folder
    os.makedirs(save_folder, exist_ok=True)
    
    print(f"Plotting {len(epochs_roi)} epochs with ROI colors...")
    
    # Plot each epoch
    for idx in range(len(epochs_roi)):
        # Create the plot
        fig = epochs_roi[idx].plot(
            n_epochs=1,
            scalings=scalings,
            n_channels=len(available_channels),
            title=f'{patient_id} - Epoch {idx+1} - {phase_name}',
            show=False
        )
        
        # Color the channel labels by ROI
        # Get the axes with channel labels
        ax = fig.axes[0]
        
        # Iterate through y-tick labels (channel names) and color them
        for label in ax.get_yticklabels():
            ch_name = label.get_text()
            if ch_name in channel_colors:
                label.set_color(channel_colors[ch_name])
                label.set_weight('bold')
        
        # Add legend for ROI colors
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=color, label=roi_name) 
                          for roi_name, color in roi_colors.items()]
        ax.legend(handles=legend_elements, loc='upper right', fontsize=8)
        
        # Save figure
        filename = f'{patient_id}_epoch_{idx+1}_{phase_name}_ROI_colored.png'
        fig.savefig(os.path.join(save_folder, filename), dpi=150, bbox_inches='tight')
        print(f"  Saved: {filename}")
        plt.close(fig)
    
    print(f"All epochs saved to {save_folder}")



In [ ]:
# ========== Usage ==========

# Single set of epochs
plot_epochs_with_roi_colors(
    epochs=pre_tic_epochs_p4_spont,
    patient=patients[3],
    patient_id='MM30',
    phase_name='spontaneous',
    save_folder='epochs_roi_colored/MM30_spontaneous'
)


In [ ]:
# Loop through all patients and all phases
for patient_id, patient_phases in patients_data.items():
    # Get corresponding patient dict from patients list
    patient_idx = list(patients_data.keys()).index(patient_id)
    patient = patients[patient_idx]
    
    print(f"\n{'='*60}")
    print(f"Processing patient: {patient_id}")
    print(f"{'='*60}")
    
    for phase_name, phase_data in patient_phases.items():
        print(f"\n--- Phase: {phase_name} ---")
        
        # Unpack based on phase type
        if phase_name == 'spontaneous':
            pre_tic_epochs, random_epochs, bad_channels = phase_data
            epochs_dict = {}
            if pre_tic_epochs is not None and len(pre_tic_epochs) > 0:
                epochs_dict['pre_tic'] = pre_tic_epochs
            if random_epochs is not None and len(random_epochs) > 0:
                epochs_dict['random'] = random_epochs
        
        else:  # imitated or suppressed
            pre_tic_cat1, pre_tic_cat2, random_epochs, bad_channels = phase_data
            category_name = phase_name  # 'imitated' or 'suppressed'
            epochs_dict = {}
            
            if pre_tic_cat1 is not None and len(pre_tic_cat1) > 0:
                epochs_dict[category_name] = pre_tic_cat1
            if pre_tic_cat2 is not None and len(pre_tic_cat2) > 0:
                epochs_dict['real'] = pre_tic_cat2
            if random_epochs is not None and len(random_epochs) > 0:
                epochs_dict['random'] = random_epochs
        
        # Plot each category
        for category_name, epochs in epochs_dict.items():
            print(f"  Plotting {category_name}: {len(epochs)} epochs")
            
            plot_raw_epochs_with_roi(
                epochs=epochs,
                patient=patient,
                patient_id=patient_id,
                phase_name=f'{phase_name}_{category_name}',
                save_folder=f'raw_epochs_norm/{patient_id}/{phase_name}/{category_name}'
            )

## Time-Frequency Analysis

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import mne

from mne import Epochs, create_info
from mne.io import RawArray
from mne.time_frequency import AverageTFRArray, EpochsTFRArray, tfr_morlet

print(__doc__)

## Analysis Per Event

Each event is plotted seperately after z-score normalization, 5 selected ROI are chosen for plotting

In [ ]:
def plot_tfr_per_epoch_per_roi(patient, epochs_dict, random_epochs, patient_id, phase_type,
                                event_key='urge', tmin=-1, tmax=0.2,
                                save_folder=None, normalization='zscore'):

    import matplotlib.pyplot as plt
    import math
    import numpy as np
    import os

    if save_folder:
        os.makedirs(save_folder, exist_ok=True)

    all_tfr_by_condition = {}
    freqs, times = None, None

    for condition_name, epochs in epochs_dict.items():
        if epochs is None or len(epochs) == 0:
            print(f"  SKIPPING {condition_name} - empty epoch array")
            continue

        epochs_event = epochs[event_key]
        n_epochs = len(epochs_event)
        all_tfr_by_condition[condition_name] = []
    # compute TFR for each epoch and ROI
        for idx in range(n_epochs):
            roi_tfr, freqs, times, _ = tfr_per_ROI_normalized(
                patient,
                epochs_event[idx],
                random_epochs,
                epoch_type='pre_tic',
                freqs=np.arange(1.0, 40.0, 2.0),
                bad_channels=[],
                normalization=normalization
            )
            all_tfr_by_condition[condition_name].append(roi_tfr)

   # Compute vmin/vmax based on 98th percentile across all conditions, epochs, and ROIs
    all_values = np.concatenate([
        data.flatten()
        for condition_tfrs in all_tfr_by_condition.values()  
        for roi_tfr in condition_tfrs                       
        for data in roi_tfr.values()                       
    ])

    abs_max = np.percentile(np.abs(all_values), 98)
    vmin, vmax = -abs_max, abs_max
    print(f"\nColor scale (98th percentile across all conditions): vmin={vmin:.2f}, vmax={vmax:.2f}")

    for condition_name, condition_tfrs in all_tfr_by_condition.items():
        print(f"\nPlotting condition: {condition_name}")

        for idx, roi_tfr in enumerate(condition_tfrs):
            fig = plot_trf_roi(
                roi_tfr, freqs, times,
                n=1, vmin=vmin, vmax=vmax,
                epoch_type='pre_tic'
            )
            fig.suptitle(f"{patient_id} - {phase_type} - {condition_name} - Epoch {idx+1}",
                        fontsize=14)

            if save_folder:
                filename = f'{patient_id}_{phase_type}_{condition_name}_epoch_{idx+1}.png'
                fig.savefig(os.path.join(save_folder, filename), dpi=150, bbox_inches='tight')
                print(f"  Saved: {filename}")
                plt.close(fig)
            else:
                plt.show()


In [ ]:
# Example usage for one patient and one phase
patient_id = "DS26"
phase = "spontaneous"
pre_tic_epochs, random_epochs, bad_channels = patients_data[patient_id][phase]
plot_tfr_per_epoch_per_roi(
    patient=patients[0], 
    epochs=pre_tic_epochs, 
    random_epochs=random_epochs, 
    patient_id=patient_id, 
    phase_type=phase,
    save_folder="tfr_plots_per_event/DS26",
    normalization='zscore'
)


In [ ]:
for patient_id, patient, patient_phases in zip(patients_data.keys(), patients, patients_data.values()):
    
    for phase_name, phase_data in patient_phases.items():
        print(f"\n--- Phase: {phase_name} ---")

        if phase_name == 'spontaneous':
            pre_tic_epochs, random_epochs, bad_channels = phase_data
            epochs_dict = {'spontaneous': pre_tic_epochs}
        
        else:  
            pre_tic_cat1, pre_tic_cat2, random_epochs, bad_channels = phase_data
            category_name = 'imitated' if phase_name == 'imitated' else 'suppressed'
            
            epochs_dict = {}
            if pre_tic_cat1 is not None:
                epochs_dict[category_name] = pre_tic_cat1
            else:
                print(f"  SKIPPING {category_name} - empty")
            
            if pre_tic_cat2 is not None:
                epochs_dict['real'] = pre_tic_cat2
            else:
                print(f"  SKIPPING real - empty")
            
            if not epochs_dict:
                print(f"  SKIPPING {phase_name} entirely - all empty")
                continue
        
        # Plot
        plot_tfr_per_epoch_per_roi(
            patient=patient,
            epochs_dict=epochs_dict,
            random_epochs=random_epochs,
            patient_id=patient_id,
            phase_type=phase_name,
            save_folder=f'tfr_per_epoch_norm/{patient_id}/{phase_name}'
        )

## Analysis Averaged Across Epochs 

In [ ]:
def plot_tfr_averaged_all_phases(patient, patient_id, patient_data, save_folder='tfr_results_all_phases'):
    """
    Plot averaged TFR for all phases of one patient using preloaded data.
    Skips empty epoch arrays (None).
    vmin/vmax based on 98th percentile across all phases and conditions.
    """
    import os
    import matplotlib.pyplot as plt
    import numpy as np

    os.makedirs(save_folder, exist_ok=True)

    # compute all TFRs first                                      
    all_roi_tfr = {}  

    for phase_name, phase_data in patient_data.items():
        print(f"\n--- Phase: {phase_name} ---")

        # Unpack based on phase type
        if phase_name == 'spontaneous':
            pre_tic_epochs, random_epochs, bad_channels = phase_data
            
            if pre_tic_epochs is None or len(pre_tic_epochs) == 0:
                print(f"  SKIPPING {phase_name} - empty epoch array")
                continue
            
            epochs_dict = {'spontaneous': pre_tic_epochs}

        else:  # imitated or suppressed
            pre_tic_cat1, pre_tic_cat2, random_epochs, bad_channels = phase_data
            category_name = phase_name  # 'imitated' or 'suppressed'
            epochs_dict = {}

            if pre_tic_cat1 is not None and len(pre_tic_cat1) > 0:
                epochs_dict[category_name] = pre_tic_cat1
            else:
                print(f"  SKIPPING {category_name} - empty epoch array")

            if pre_tic_cat2 is not None and len(pre_tic_cat2) > 0:
                epochs_dict['real'] = pre_tic_cat2
            else:
                print(f"  SKIPPING real - empty epoch array")

            if not epochs_dict:
                print(f"  SKIPPING {phase_name} entirely - all epoch arrays empty")
                continue

        # Compute TFR for each category
        for category_name, epochs in epochs_dict.items():
            print(f"  Computing TFR for {category_name}: {len(epochs)} epochs")

            roi_tfr, freqs, times, n_epochs = tfr_per_ROI_normalized(
                patient,
                epochs,
                epoch_type='pre_tic',
                random_epochs=random_epochs,
                bad_channels=bad_channels,
                normalization='zscore'
            )

            all_roi_tfr[(phase_name, category_name)] = (roi_tfr, freqs, times, n_epochs)

    if not all_roi_tfr:
        print("No TFR results computed - skipping")
        return

    # compute vmin/vmax across ALL phases and conditions          
    all_values = np.concatenate([
        data.flatten()
        for roi_tfr, freqs, times, n_epochs in all_roi_tfr.values()
        for data in roi_tfr.values()
    ])

    abs_max = np.percentile(np.abs(all_values), 98)
    vmin, vmax = -abs_max, abs_max
    print(f"\nColor scale (98th percentile across all phases): vmin={vmin:.2f}, vmax={vmax:.2f}")

    #  plot all TFRs with consistent color scale                   
    for (phase_name, category_name), (roi_tfr, freqs, times, n_epochs) in all_roi_tfr.items():
        fig = plot_trf_roi(roi_tfr, freqs, times, n_epochs, epoch_type='pre_tic', vmin=vmin, vmax=vmax)

        filename = f"{patient_id}_{phase_name}_{category_name}_TFR_n{n_epochs}.png"
        fig.savefig(
            os.path.join(save_folder, filename),
            dpi=300,
            bbox_inches="tight"
        )
        print(f"  Saved: {filename}")
        plt.close(fig)




In [ ]:
# Single patient, all phases
plot_tfr_averaged_all_phases(
    patient=patients[1],
    patient_id='BB28',
    phases_to_plot=['spontaneous', 'imitated', 'suppressed'],
    save_folder='tfr_results_all_phases_2/BB28'
)

In [ ]:
# Loop through all patients
patient_id = ["DS26", "BB28", "BC29", "MM30", "SC31"]
# ========== Loop through all patients using preloaded patients_data ==========
for patient_id, patient_phases in patients_data.items():
    # Get corresponding patient dict from patients list
    patient_idx = list(patients_data.keys()).index(patient_id)
    
    plot_tfr_averaged_all_phases(
        patient=patients[patient_idx],
        patient_id=patient_id,
        patient_data=patient_phases,
        save_folder=f'tfr_results_all_phases_norm/{patient_id}'
    )